# ASSIGNMENT 1 - Question 5 - ECG Arrhythmia Model on a Wearable
### Alumno: Martín Andrés Gutiérrez López
Topics Batch Normalization, Layer Normalization

In [2]:
# ==============================================================================
# Question 5: ECG Arrhythmia Model on a Wearable (HeartBand Case)
# Verification of Batch Normalization, Layer Normalization, and Batch Size 1 Issue
# ==============================================================================

import numpy as np
import torch
import torch.nn as nn

BOLD = "\033[1m"
BOLD_BLUE = "\033[1;34m"
RESET = "\033[0m"

# ------------------------------------------------------------------------------
# Input Mini-batch X: 4 samples x 3 features
# ------------------------------------------------------------------------------
X_np = np.array([[2, 4, 6], [4, 8, 2], [6, 0, 4], [8, 4, 0]], dtype=float)

# ==============================================================================
# Part (a): Batch Normalization on feature f1 (Column index 0)
# ==============================================================================
f1 = X_np[:, 0]  # [2, 4, 6, 8]

# Calculate batch mean and variance across samples (N = 4)
mu_B = np.mean(f1)
var_B = np.var(f1)  # Population variance (divide by N)
std_B = np.sqrt(var_B)

# Normalized values (gamma=1, beta=0, eps=0)
f1_norm = (f1 - mu_B) / std_B

print(f"\n{BOLD_BLUE}")
print("=" * 70)
print("PART (a): BATCH NORMALIZATION ON FEATURE f1")
print("=" * 70,f"{RESET}")
print(f"Batch Mean (mu_B)           : {mu_B:.4f} (Exact: 5.0)")
print(f"Batch Variance (sigma_B^2)  : {var_B:.4f} (Exact: 5.0)")
print(f"Standard Deviation          : {std_B:.4f} (Exact: sqrt(5) ≈ 2.2361)")
print("Normalized values for f1    :")
for i, val in enumerate(f1_norm):
  print(f"  Sample x{i+1}: {val:+.4f}")

# ==============================================================================
# Part (b): Layer Normalization on sample x1 (Row index 0)
# ==============================================================================
x1 = X_np[0, :]  # [2, 4, 6]

# Calculate layer mean and variance across features (D = 3)
mu_L = np.mean(x1)
var_L = np.var(x1)  # Population variance (divide by D)
std_L = np.sqrt(var_L)

# Normalized values (gamma=1, beta=0, eps=0)
x1_norm = (x1 - mu_L) / std_L

print(f"\n{BOLD_BLUE}")
print("=" * 70)
print("PART (b): LAYER NORMALIZATION ON SAMPLE x1")
print("=" * 70,f"{RESET}")
print(f"Sample Mean (mu)            : {mu_L:.4f} (Exact: 4.0)")
print(f"Sample Variance (sigma^2)   : {var_L:.4f} (Exact: 8/3 ≈ 2.6667)")
print(f"Standard Deviation          : {std_L:.4f} (Exact: sqrt(8/3) ≈ 1.6330)")
print("Normalized values for x1    :")
for j, val in enumerate(x1_norm):
  print(f"  Feature f{j+1}: {val:+.4f}")

# ==============================================================================
# Part (c): PyTorch Simulation of BatchNorm Failure with Batch Size = 1
# ==============================================================================
print(f"\n{BOLD_BLUE}")
print("=" * 70)
print("PART (c): PYTORCH DEMONSTRATION OF BATCH SIZE 1 FAILURE")
print("=" * 70,f"{RESET}")

# Create a BatchNorm1d layer without learnable affine parameters (gamma=1, beta=0)
bn = nn.BatchNorm1d(num_features=3, affine=False)

# 1. Training step with batch size = 4 to populate running stats
X_tensor = torch.tensor(X_np, dtype=torch.float32)
bn.train()
_ = bn(X_tensor)

print(f"Pre-computed Running Mean      : {bn.running_mean.numpy()}")
print(f"Pre-computed Running Variance  : {bn.running_var.numpy()}")

# 2. Incoming single sample at inference time: x1
single_sample = torch.tensor([[2.0, 4.0, 6.0]], dtype=torch.float32)

# Case A: Incorrect deployment (left in training mode)
bn.train()
try:
  output_train_mode = bn(single_sample)
  print(f"Output in train mode (batch size 1): {output_train_mode.numpy()}")
except ValueError as e:
  print(f"PyTorch Error in train mode with batch size 1: {e}")

# Case B: Correct deployment (switched to eval mode)
bn.eval()
output_eval_mode = bn(single_sample)
print(f"Output in eval mode (using running stats): {output_eval_mode.numpy()}")



PART (a): BATCH NORMALIZATION ON FEATURE f1
Batch Mean (mu_B)           : 5.0000 (Exact: 5.0)
Batch Variance (sigma_B^2)  : 5.0000 (Exact: 5.0)
Standard Deviation          : 2.2361 (Exact: sqrt(5) ≈ 2.2361)
Normalized values for f1    :
  Sample x1: -1.3416
  Sample x2: -0.4472
  Sample x3: +0.4472
  Sample x4: +1.3416


PART (b): LAYER NORMALIZATION ON SAMPLE x1
Sample Mean (mu)            : 4.0000 (Exact: 4.0)
Sample Variance (sigma^2)   : 2.6667 (Exact: 8/3 ≈ 2.6667)
Standard Deviation          : 1.6330 (Exact: sqrt(8/3) ≈ 1.6330)
Normalized values for x1    :
  Feature f1: -1.2247
  Feature f2: +0.0000
  Feature f3: +1.2247


PART (c): PYTORCH DEMONSTRATION OF BATCH SIZE 1 FAILURE
Pre-computed Running Mean      : [0.5 0.4 0.3]
Pre-computed Running Variance  : [1.5666666 1.9666667 1.5666666]
PyTorch Error in train mode with batch size 1: Expected more than 1 value per channel when training, got input size torch.Size([1, 3])
Output in eval mode (using running stats): [[1.1983993 2.